In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

split = "valid"
model = "full_model"
cells = "812"

TRUE_PATH = Path("/home/jovyan/dpanc/benchmarking/data/borzoi_all_ids_qnorm_matrix.csv")
# PRED_PATH = Path("/home/jovyan/dpanc/benchmarking/graund_true_comparison/data/gena_predictions_ontology_mean_for_alphagenome.csv")
PRED_PATH = Path(f"/home/jovyan/dpanc/benchmarking/GENA_LM/predictions_results/{model}/gena_lm_{split}_json{cells}_predictions.csv")
DATA_ROOT = Path("/home/jovyan/dpanc/benchmarking/data")

GENE_FILES = [
    DATA_ROOT / "human.test.forward.csv",
    DATA_ROOT / "human.test.reverse.csv",
    DATA_ROOT / "human.valid.forward.csv",
    DATA_ROOT / "human.valid.reverse.csv",
]


In [6]:
def load_true_df(path):
    # GT file has tracks/cell IDs as rows and genes as columns:
    # id, original_id, targets_identifier_base, strand_specificity, ENSG...
    true_df = pd.read_csv(path)
    if "gene_id" in true_df.columns:
        return true_df

    gene_cols = [c for c in true_df.columns if str(c).startswith(("ENSG", "ENSMUSG"))]
    true_df = (
        true_df.set_index("id")[gene_cols]
        .T
        .reset_index()
        .rename(columns={"index": "gene_id"})
    )
    return true_df


def load_split_map(gene_files):
    tables = []
    for path in gene_files:
        df = pd.read_csv(path, sep="\t", usecols=["gene_id", "split"])
        df["source_file"] = path.name
        tables.append(df)
    split_df = pd.concat(tables, ignore_index=True).drop_duplicates("gene_id")
    return split_df


def align_dataframes(true, pred):
    common_genes = sorted(set(true["gene_id"]).intersection(pred["gene_id"]))
    common_cols = sorted(set(true.columns[1:]).intersection(pred.columns[1:]))

    true_aligned = (
        true[true["gene_id"].isin(common_genes)]
        .sort_values("gene_id")
        [["gene_id"] + common_cols]
        .reset_index(drop=True)
    )
    pred_aligned = (
        pred[pred["gene_id"].isin(common_genes)]
        .sort_values("gene_id")
        [["gene_id"] + common_cols]
        .reset_index(drop=True)
    )
    return true_aligned, pred_aligned


def compute_metrics_for_split(true_log, pred_aligned, split_name):
    cell_cols = true_log.columns[1:]

    per_cell = []
    for col in cell_cols:
        true_vec = true_log[col].to_numpy(dtype=np.float64)
        pred_vec = pred_aligned[col].to_numpy(dtype=np.float64)
        corr = np.nan if np.std(true_vec) == 0 or np.std(pred_vec) == 0 else float(np.corrcoef(true_vec, pred_vec)[0, 1])
        per_cell.append({"split": split_name, "cell_id": col, "corr_across_genes": corr, "genes_evaluated": len(true_vec)})

    per_gene = []
    for i in range(true_log.shape[0]):
        gene_id = true_log.iloc[i]["gene_id"]
        true_vec = true_log.iloc[i][cell_cols].to_numpy(dtype=np.float64)
        pred_vec = pred_aligned.iloc[i][cell_cols].to_numpy(dtype=np.float64)
        corr = np.nan if np.std(true_vec) == 0 or np.std(pred_vec) == 0 else float(np.corrcoef(true_vec, pred_vec)[0, 1])
        per_gene.append({"split": split_name, "gene_id": gene_id, "corr_across_cell_ids": corr, "cell_ids_evaluated": len(cell_cols)})

    per_cell_df = pd.DataFrame(per_cell)
    per_gene_df = pd.DataFrame(per_gene)
    summary = {
        "split": split_name,
        "genes": true_log.shape[0],
        "cells": len(cell_cols),
        "corr_genes": per_cell_df["corr_across_genes"].mean(skipna=True),
        "corr_cells": per_gene_df["corr_across_cell_ids"].mean(skipna=True),
    }
    return summary, per_cell_df, per_gene_df


In [7]:
true_df = load_true_df(TRUE_PATH)
pred_df = pd.read_csv(PRED_PATH)
split_map = load_split_map(GENE_FILES)

print("true_df shape:", true_df.shape)
print("pred_df shape:", pred_df.shape)
print("split counts in original files:")
display(split_map["split"].value_counts())

true_aligned, pred_aligned = align_dataframes(true_df, pred_df)

split_map = split_map[split_map["gene_id"].isin(pred_aligned["gene_id"])].copy()
pred_aligned = pred_aligned.merge(split_map[["gene_id", "split"]], on="gene_id", how="left")
true_aligned = true_aligned.merge(split_map[["gene_id", "split"]], on="gene_id", how="left")

pred_aligned = pred_aligned[["gene_id", "split"] + [c for c in pred_aligned.columns if c not in ["gene_id", "split"]]]
true_aligned = true_aligned[["gene_id", "split"] + [c for c in true_aligned.columns if c not in ["gene_id", "split"]]]

print("common genes:", pred_aligned.shape[0])
print("common cell IDs:", pred_aligned.shape[1] - 2)
print("prediction genes by split:")
display(pred_aligned["split"].value_counts(dropna=False))
display(pred_aligned.iloc[:3, :8])
display(true_aligned.iloc[:3, :8])


true_df shape: (24759, 434)
pred_df shape: (3038, 813)
split counts in original files:


split
valid    3038
test     2781
Name: count, dtype: int64

common genes: 3038
common cell IDs: 430
prediction genes by split:


split
valid    3038
Name: count, dtype: int64

,gene_id,split,ENCFF003QOJ,ENCFF009MEF,ENCFF010XLY,ENCFF011DHD,ENCFF013HFB,ENCFF013YPF
0,ENSG00000001617.11,valid,0.617188,2.406250,2.843750,0.789062,0.851562,0.871094
1,ENSG00000002016.17,valid,2.031250,0.542969,0.240234,0.128906,0.136719,0.186523
2,ENSG00000002549.12,valid,1.648438,2.765625,3.281250,2.437500,2.453125,2.953125


,gene_id,split,ENCFF003QOJ,ENCFF009MEF,ENCFF010XLY,ENCFF011DHD,ENCFF013HFB,ENCFF013YPF
0,ENSG00000001617.11,valid,5.043247,9.087844,10.770515,1.183750,0.992297,14.138036
1,ENSG00000002016.17,valid,34.302836,14.985961,14.599579,7.120480,9.169860,4.292826
2,ENSG00000002549.12,valid,7.010060,10.133219,18.838646,11.843848,13.458842,27.654300


In [8]:
# Convert GT matrix values to log2(x + 1), like in the previous AlphaGenome notebook.

true_log = true_aligned.copy()
expr_cols = [c for c in true_log.columns if c not in ["gene_id", "split"]]
true_log[expr_cols] = np.log2(true_log[expr_cols].astype(float) + 1)

display(true_log.iloc[:3, :8])


,gene_id,split,ENCFF003QOJ,ENCFF009MEF,ENCFF010XLY,ENCFF011DHD,ENCFF013HFB,ENCFF013YPF
0,ENSG00000001617.11,valid,2.595324,3.334546,3.557105,1.126808,0.994432,3.920106
1,ENSG00000002016.17,valid,5.141712,3.998734,3.963435,3.021565,3.346228,2.404038
2,ENSG00000002549.12,valid,3.001813,3.476799,4.310242,3.683006,3.853880,4.840680


In [9]:
summary_rows = []
per_cell_tables = []
per_gene_tables = []

for split_name in ["test", "valid", "all"]:
    if split_name == "all":
        true_part = true_log.drop(columns=["split"])
        pred_part = pred_aligned.drop(columns=["split"])
    else:
        true_part = true_log[true_log["split"].eq(split_name)].drop(columns=["split"]).reset_index(drop=True)
        pred_part = pred_aligned[pred_aligned["split"].eq(split_name)].drop(columns=["split"]).reset_index(drop=True)

    if true_part.empty:
        print("No genes for split:", split_name)
        continue

    summary, per_cell_df, per_gene_df = compute_metrics_for_split(true_part, pred_part, split_name)
    summary_rows.append(summary)
    per_cell_tables.append(per_cell_df)
    per_gene_tables.append(per_gene_df)

summary_df = pd.DataFrame(summary_rows)
per_cell_df = pd.concat(per_cell_tables, ignore_index=True)
per_gene_df = pd.concat(per_gene_tables, ignore_index=True)

display(summary_df)


No genes for split: test


,split,genes,cells,corr_genes,corr_cells
0,valid,3038,430,0.772625,0.393012
1,all,3038,430,0.772625,0.393012
